# Popularity Baseline Model 
We defined an activity as popular based on the number of unique postive (rating >= 4) user interactions. <br>
This baseline model establishes a simple benchmark for model performance. <br>  

In [65]:
import pandas as pd 

train_df = pd.read_csv("../data/processed/train60.csv")
train_df.head()

,user_id,gmap_id,rating,time
0,100000012795753291799,0x89c25d977025692d:0x378d24e13eb8f34d,4,1512264700675
1,100000012795753291799,0x89c258e28c304997:0xfcafe4e7ce35ee8c,4,1518768933898
2,100000012795753291799,0x89c2590f34e21a53:0x749079380908a830,4,1519751339673
3,100000012795753291799,0x89c25999b9c6eff9:0x4b98becb736928d8,4,1569701541136
4,100000012795753291799,0x89c25cfd883fc003:0xb4ba2be70609711b,4,1570596729750


In [66]:
# positive interaction means rating >= 4
positive_interactions = train_df[train_df["rating"] >= 4]

popularity = (
    positive_interactions
    .groupby("gmap_id")["user_id"]
    .nunique()  
    .reset_index(name="popularity_score")
    .sort_values("popularity_score", ascending=False)
)

popularity.head(10)

,gmap_id,popularity_score
23422,0x89c25e4bc3a2b003:0x1b2445a846ea9a58,4771
29827,0x89c279b142e9dc41:0xf3bfe6b9d563bad,4614
18647,0x89c25b1087680443:0x178bf7c600df7a98,4435
7464,0x89c258f55fbea537:0x49d5f074061cac4e,4430
26605,0x89c2607fc22166fb:0x133d2c68e8326ce1,4372
1392,0x89c24343ab3c4adf:0xfb7370c49a9db106,4366
13496,0x89c2599f2db306bb:0x2a5c7f9eea6938a0,4279
25926,0x89c25fe0b147d195:0xf55b95d4f099763b,4205
28525,0x89c2644bfae67439:0x7e77a9e355f9aa3,4134
14036,0x89c259a6bd917da7:0x79997d3983197d7c,3952


In [67]:
#popularity.to_csv("../data/processed/popularity.csv", index=False)

"popularity score" is the number of unique users who had a positive experince with each gmapid location. 

In [68]:
seen_by_user = (
    train_df
    .groupby("user_id")
    .apply(set)
    .to_dict()        
)

def recommend_popularity(user_id, k=10):

    seen = seen_by_user.get(user_id, set())

    recommendations = popularity[
        ~popularity["gmap_id"].isin(seen)
    ].head(k).copy()

    recommendations["user_id"] = user_id
    recommendations["rank"] = range(1, len(recommendations) + 1)

    recommendations = recommendations.rename(
        columns={"popularity_score": "score"}
    )

    return recommendations[
        ["user_id", "gmap_id", "score", "rank"]
    ]

In [69]:
# evaluation functions for recall, ndcg, hitrate
import math

def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    hits = len(set(recommended_k) & set(relevant))

    return hits / len(relevant)


def ndcg_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    # Actual DCG
    dcg = 0
    for i, item in enumerate(recommended_k):
        if item in relevant:
            dcg += 1 / math.log2(i + 2)

    # Best possible DCG
    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / math.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg


def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]

    if len(relevant) == 0:
        return None

    hits = set(recommended_k) & set(relevant)

    if len(hits) > 0:
        return 1
    else:
        return 0


def calculate_metrics(recommended, relevant, k=10):

    recall = recall_at_k(recommended, relevant, k)
    hit_rate = hit_rate_at_k(recommended, relevant, k)
    ndcg = ndcg_at_k(recommended, relevant, k)

    return {
        f"Recall@{k}": recall,
        f"HitRate@{k}": hit_rate,
        f"NDCG@{k}": ndcg
    }

In [70]:
test_df = pd.read_csv("../data/processed/test20.csv")
test_df.head()

,user_id,gmap_id,rating,time
0,100000012795753291799,0x89c25a4c43c5a19f:0x3f6401582b49c95c,4,1617057156812
1,100000036464530353686,0x89c2f5b9faaed00f:0x8b1a77bb24200925,5,1557705305074
2,100000036464530353686,0x89c258ff48e75613:0x1de767bc6d9ae0c,5,1557705390025
3,100000036464530353686,0x89c2f662a75ebad5:0x54bfc08ac601aa8d,5,1561338245436
4,100000036464530353686,0x89c28c633b025067:0x8dd7fd3725821408,5,1561338251913


In [71]:
import random
test_pos = test_df[test_df["rating"] >= 4]

eligible_users = list(
    set(train_df["user_id"]) &
    set(test_pos["user_id"])
)

random.seed(42)

sample_size = min(1000, len(eligible_users))
evaluation_users = random.sample(eligible_users, sample_size)

relevant_by_user = (
    test_pos
    .groupby("user_id")["gmap_id"]
    .apply(set)
    .to_dict()
)
hit_scores, recall_scores, ndcg_scores = [], [], []

for user_id in evaluation_users:

    recs = recommend_popularity(user_id, k=10)
    recommended = recs["gmap_id"].tolist()

    relevant = set(
        test_pos[
            test_pos["user_id"] == user_id
        ]["gmap_id"]
    )

    recall_scores.append(
        recall_at_k(recommended, relevant, 10)
    )

    hit_scores.append(
        hit_rate_at_k(recommended, relevant, 10)
    )

    ndcg_scores.append(
        ndcg_at_k(recommended, relevant, 10)
    )

avg_recall = sum(recall_scores) / len(recall_scores)
avg_hit = sum(hit_scores) / len(hit_scores)
avg_ndcg = sum(ndcg_scores) / len(ndcg_scores)

print("Recall:", avg_recall)
print("Hit Rate:", avg_hit)
print("NDCG:", avg_ndcg)

Recall: 0.010979304029304028
Hit Rate: 0.024
NDCG: 0.00576101567509613
